# YouTube → Google Drive Downloader

Downloads YouTube videos/playlists straight to your Google Drive using `yt-dlp`. Run the cells top to bottom.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title 📥 YouTube Videos to Google Drive { display-mode: "form" }
# @markdown Fill in the details below and click ▶ Run.

import os
import subprocess
from IPython.display import HTML, display

# @markdown **YouTube URL (video or playlist):**
yt_url = '' # @param {type:"string"}

# @markdown **Save to folder:**
save_to = '/content/drive/MyDrive/YouTube Downloads/' # @param {type:"string"}

# @markdown **Video quality:**
quality = "1080p" # @param ["4K", "1080p", "720p", "480p", "360p", "Audio Only (MP3)"]

# @markdown **Download entire playlist?**
is_playlist = False # @param {type:"boolean"}

# @markdown **Playlist range (e.g. 1-5, leave blank for all):**
playlist_range = '' # @param {type:"string"}

# @markdown **Path to cookies.txt (optional, needed for age-restricted videos or "Sign in to confirm you\'re not a bot"):**
cookie_file_path = '' # @param {type:"string"}

# @markdown **List available formats instead of downloading?**
list_formats = False # @param {type:"boolean"}

# -- logic -------------------------

status_display = display(HTML(""), display_id='install_status')
progress_display = display(HTML(""), display_id='download_progress')

def show_status(text, color="#4ecca3"):
    status_display.update(HTML(f"""
    <div style="background: linear-gradient(135deg, #1a1a2e, #0f3460);
                padding: 15px 25px; border-radius: 10px; margin-bottom:10px;">
      <p style="color:{color}; font-size:1.1em; margin:0;">{text}</p>
    </div>
    """))

if not yt_url.strip():
    show_status("⚠️ Please enter a YouTube URL above and run again.", color="#e94560")
else:
    show_status("⏳ Installing / updating yt-dlp, please wait...")
    # 'yt-dlp[default]' bundles the yt-dlp-ejs scripts needed to solve YouTube's
    # JS challenge. Without them yt-dlp silently falls back to thumbnail-only formats.
    subprocess.run(['pip', 'install', '-q', '-U', 'yt-dlp[default]'], capture_output=True)
    # Deno is the JS runtime yt-dlp uses to solve that challenge; Colab doesn't ship it.
    if subprocess.run(['which', 'deno'], capture_output=True).returncode != 0:
        subprocess.run(['bash', '-c', 'curl -fsSL https://deno.land/install.sh | sh -s -- -y'], capture_output=True)
    deno_bin = os.path.expanduser('~/.deno/bin')
    if deno_bin not in os.environ['PATH']:
        os.environ['PATH'] = deno_bin + os.pathsep + os.environ['PATH']
    import yt_dlp
    show_status("✅ yt-dlp ready. Starting...")

    os.makedirs(save_to, exist_ok=True)

    quality_map = {
        '4K':               'bestvideo[height<=2160]+bestaudio/best[height<=2160]/best',
        '1080p':            'bestvideo[height<=1080]+bestaudio/best[height<=1080]/best',
        '720p':             'bestvideo[height<=720]+bestaudio/best[height<=720]/best',
        '480p':             'bestvideo[height<=480]+bestaudio/best[height<=480]/best',
        '360p':             'bestvideo[height<=360]+bestaudio/best[height<=360]/best',
        'Audio Only (MP3)': 'bestaudio/best',
    }
    selected_format = quality_map[quality]

    def download_progress_hook(d):
        if d['status'] == 'downloading':
            total_bytes = d.get('total_bytes') or d.get('total_bytes_estimate')
            downloaded_bytes = d.get('downloaded_bytes', 0)
            percent = (downloaded_bytes / total_bytes * 100) if total_bytes else 0
            speed = d.get('speed')
            eta = d.get('eta')
            speed_str = f"{speed/1024/1024:.2f} MiB/s" if speed else "N/A"
            eta_str = f"{eta}s" if eta else "N/A"
            progress_display.update(HTML(f"""
            <div style="background: linear-gradient(135deg, #0f3460, #1a1a2e);
                        padding: 10px 15px; border-radius: 8px; margin-top: 10px; color: #a8b2d8;">
              <p style="margin: 0; font-size: 1.1em;">⏳ Downloading: {d.get('filename', 'Unknown file')}</p>
              <div style="width: 100%; background-color: #333; border-radius: 5px; height: 20px; margin-top: 5px;">
                <div style="width: {percent:.2f}%; background-color: #4ecca3; height: 100%; border-radius: 5px;
                            text-align: center; line-height: 20px; color: #1a1a2e;">
                  {percent:.2f}%
                </div>
              </div>
              <p style="margin: 5px 0 0 0; font-size: 0.9em;">Speed: {speed_str} | ETA: {eta_str}</p>
            </div>
            """))
        elif d['status'] == 'finished':
            progress_display.update(HTML(""))

    if list_formats:
        ydl_opts = {
            'listformats': True,
            'quiet': False,
            'no_warnings': False,
            'noplaylist': not is_playlist,
        }
    else:
        ydl_opts = {
            'format': selected_format,
            'outtmpl': os.path.join(save_to, '%(title)s.%(ext)s'),
            'merge_output_format': 'mp4',
            'noplaylist': not is_playlist,
            'quiet': True,
            'no_warnings': True,
            'progress_hooks': [download_progress_hook],
            'retries': 10,
            'fragment_retries': 10,
            'continuedl': True,
            'retry_sleep_functions': {
                'http': lambda n: 5 * n,
                'fragment': lambda n: 3 * n,
            },
        }
        if quality == 'Audio Only (MP3)':
            ydl_opts['postprocessors'] = [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'mp3',
                'preferredquality': '192',
            }]

    # Applies whether listing formats or downloading, so 'list formats' previews
    # only the items you actually asked for instead of the whole playlist.
    if is_playlist and playlist_range:
        ydl_opts['playlist_items'] = playlist_range

    if cookie_file_path and os.path.exists(cookie_file_path):
        ydl_opts['cookiefile'] = cookie_file_path
    elif cookie_file_path:
        print(f"Warning: cookies file not found at {cookie_file_path}. Continuing without cookies.")

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            if list_formats:
                ydl.extract_info(yt_url, download=False)
                show_status("✅ Formats listed above.")
            else:
                # NOTE: this must be ydl.download(), not extract_info(download=False),
                # otherwise yt-dlp only fetches metadata and never saves the file.
                ydl.download([yt_url])
                show_status(f"✅ Done! Saved to {save_to}")
    except Exception as e:
        show_status(f"❌ Error: {e}", color="#e94560")

## Optional: merge cookies if you exported `youtube.com` and `google.com` cookies separately

Some cookie-export extensions save YouTube and Google login cookies into two separate files.
`yt-dlp` needs one combined file. This cell merges them and converts JSON exports to the
Netscape format `yt-dlp` expects. Skip this if your `cookies.txt` already has everything in one file.

In [ ]:
import json
import os

# @markdown **Path to your youtube.com cookie file:**
youtube_cookies_path = '/content/drive/MyDrive/youtube_cookies.txt' # @param {type:"string"}

# @markdown **Path to your google.com cookie file:**
google_cookies_path = '/content/drive/MyDrive/google_cookies.txt' # @param {type:"string"}

# @markdown **Path for the combined output file (paste this into "Path to cookies.txt" above):**
combined_cookies_path = '/content/drive/MyDrive/combined_cookies.txt' # @param {type:"string"}

def _json_to_netscape(cookies):
    lines = ['# Netscape HTTP Cookie File']
    for c in cookies:
        expiry = int(c.get('expirationDate', c.get('expires', 0)))
        if expiry > 253402300000:  # ms instead of seconds
            expiry //= 1000
        domain = c.get('domain', '')
        host_only = c.get('hostOnly', False)
        if not host_only and not domain.startswith('.'):
            domain = '.' + domain
        flag = 'FALSE' if host_only else 'TRUE'
        path = c.get('path', '/')
        secure = 'TRUE' if c.get('secure', False) else 'FALSE'
        name = c.get('name', '')
        value = c.get('value', '') or ''
        lines.append(f"{domain}\t{flag}\t{path}\t{secure}\t{expiry}\t{name}\t{value}")
    return '\n'.join(lines)

def _read_cookie_file(path):
    if not os.path.exists(path):
        print(f"Warning: not found: {path}")
        return ""
    content = open(path, 'r').read()
    try:
        return _json_to_netscape(json.loads(content))
    except json.JSONDecodeError:
        return content  # already Netscape/plain text

combined = [c for c in (_read_cookie_file(youtube_cookies_path), _read_cookie_file(google_cookies_path)) if c]

if combined:
    with open(combined_cookies_path, 'w') as f:
        f.write('\n'.join(combined))
    print(f"Combined cookies written to: {combined_cookies_path}")
    print("Paste this path into the \"Path to cookies.txt\" field in the downloader cell above.")
else:
    print("Nothing to combine — check the file paths.")

## Troubleshooting

**"Sign in to confirm you're not a bot"**
- Re-export cookies while logged into YouTube in an **incognito window** right before running this notebook — stale cookies are the most common cause.
- Make sure the install step above just ran — YouTube changes frequently and old `yt-dlp` versions break first.
- Avoid running through a VPN or datacenter proxy; those IPs get flagged more often.
- If it still fails, try again a bit later — this is usually YouTube-side rate limiting, not a bug in this notebook.

**Only thumbnail/storyboard formats (`sb0`, `sb1`…) show up, no real video**
- This means yt-dlp couldn't solve YouTube's JS challenge. The downloader cell now installs `yt-dlp[default]` and the Deno JS runtime automatically to fix this — just re-run it once from the top so the install actually happens before the download starts.
- If it still happens after that, your Colab runtime may be blocking the Deno installer; open a Terminal (or a new code cell) and run `!curl -fsSL https://deno.land/install.sh | sh` manually, then restart the runtime and try again.